In [7]:
# Dealing with performanc:e decline after deployment:
# explore data closer to the decision boundary (inside the margin) of SVM model if such points contribute to the model's poor performance


In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics import make_scorer, f1_score, roc_auc_score, accuracy_score, log_loss
from sklearn.metrics import  precision_score, recall_score, classification_report,confusion_matrix, balanced_accuracy_score
import joblib
from pathlib import Path
import sys
import os

In [9]:
def find_project_root():
    """
    Search parent directories for ROOT folder (that contains src folder)`.
    add it to the sys path
    """
    current_path = Path.cwd()  # start from current working directory
    for parent in [current_path] + list(current_path.parents):
        if (parent / 'src').exists():
            #return (parent / 'src').resolve()
            return (parent).resolve()
    raise FileNotFoundError(f"src folder not found in any parent directory.")

# Add root directory to system path
ROOT_DIR =  find_project_root()
print("Project root directory:", ROOT_DIR)
sys.path.append(f"{ROOT_DIR}")

Project root directory: /gpfs/exfel/data/scratch/majida/Noble_Gas_Instability/New_analysis-aug-2025/git_repo_for_paper/VacFault


In [10]:
from src.config  import datasets_dir
from src.config import results_dir
from src.config import models_dir
from src.config import datasets
from src.config import feature_col
from src.config import ml_models_svm
from src.config import imbalance_learn_methods_svm
from src.config import kernel_method_svm
from src.config import random_state

In [11]:
# step1: Goal:
#   For each trained model, sweep a rnage of decision‑score thresholds
#   - Data samples whose scores fall inside the threshold are sent for
#     manual labeling.
#   - The cost of that manual labeling is recorded.
#   - For the remaining points (outside the threshold) we compute
#     the F1‑score of the model’s predictions.
    

In [12]:
out_dir = results_dir / 'post_deployment_performance_handling' # directory to save results
out_dir.mkdir(parents=True, exist_ok=True)

In [13]:
svm_margin_pos = [x for x in np.round(np.arange(0.1,0.6,0.1),1)]
svm_margin_neg = list(map(lambda x: -x,svm_margin_pos))

In [14]:
# Load deployment data subset
dataset = 'deployment_data'
data_dir = os.path.join(datasets_dir,dataset)
df = pd.read_hdf(f"{data_dir}/df_deployment.h5")


for ml_model in ml_models_svm:
    for imbalance_method in imbalance_learn_methods_svm:
        current_model_dir = f"{models_dir}/{ml_model}_{imbalance_method}/"

        # Load pipeline parameters
        pipeline_name = f"{current_model_dir}pipeline.pkl"
        pipeline = joblib.load(pipeline_name)

        # Separate features vs.labels
        X = df[feature_col]
        y = df['labels']
        y = y.astype(int)

        # Evaluate performance
        #y_pred = pipeline.predict(X)
        #precision = precision_score(y, y_pred, average=None)
        #recall = recall_score(y, y_pred, average=None)
        #f1 = f1_score(y, y_pred, average=None)
        #decision_scores = pipeline.decision_function(X)

        # create a dataframe to save results
        df_results = pd.DataFrame(columns=['margin_threshold',
                                  'total_samples_in_X_inside_margin', 'total_samples_in_X_outside_margin',
                                   'total_faults_in_X_inside_margin', 'total_faults_in_X_outside_margin',
                                   'precision_class_1_X_outside_margin','recall_class_1_X_outside_margin',
                                    'f1_class_1_X_outside_margin',
                                   #'total_number_misclassified_samples_outside_margin',
                                  ])

        # compute decision score of all data samples
        X_pred = X.copy()
        X_pred['decision_score'] = pipeline.decision_function(X)
        for margin_neg, margin_pos in zip(svm_margin_neg, svm_margin_pos):
            #print(margin_neg,margin_pos)
            
            # divide data into two subsets (X: X_inside, X_outside)   
            
            X_pred['is_inside_margin'] =X_pred['decision_score'].apply(lambda x_: 1 if x_>=margin_neg and x_<=margin_pos else 0)
            index_inside_margin = X_pred[X_pred['is_inside_margin']==1].index
            index_outside_margin = X_pred[X_pred['is_inside_margin']==0].index
            #print(inside_margin_index.shape, outside_margin_index.shape)

            X_inside_margin = X.copy().loc[index_inside_margin]
            y_inside_margin = y.copy().loc[index_inside_margin]
            X_outside_margin = X.copy().loc[index_outside_margin]
            y_outside_margin = y.copy().loc[index_outside_margin]

            if X_inside_margin.empty == True:
                continue
            if X_outside_margin.empty == True:
                continue

            # compute performance metrics for X_outside
            y_pred = pipeline.predict(X_outside_margin)
            y_true = y_outside_margin
            precision_X_outside_margin = precision_score(y_true, y_pred, average=None)
            recall_X_outside_margin = recall_score(y_true, y_pred, average=None)
            f1_X_outside_margin = f1_score(y_true, y_pred, average=None)
            #number_misclassified_samples_outside_margin = np.sum(y_true!=y_pred)

            new_row = pd.DataFrame({
                'margin_threshold': [(margin_neg,margin_pos)],
                'total_samples_in_X_inside_margin' : [X_inside_margin.shape[0]],
                'total_samples_in_X_outside_margin' : [X_outside_margin.shape[0]],
                'total_faults_in_X_inside_margin' : [y_inside_margin.value_counts().get(1, 0)], 
                'total_faults_in_X_outside_margin': [y_outside_margin.value_counts().get(1, 0)],
                'precision_class_1_X_outside_margin': [precision_X_outside_margin[1]],
                'recall_class_1_X_outside_margin': [recall_X_outside_margin[1]], 
                'f1_class_1_X_outside_margin' : [f1_X_outside_margin[1]],
                #'total_number_misclassified_samples_outside_margin': [number_misclassified_samples_outside_margin],
                  })
            df_results = pd.concat([df_results, new_row])
        
        #save data
        fname = f'{out_dir}/results_{ml_model}_{imbalance_method}.h5'
        df_results.to_hdf(fname, key="df",mode='w')
        print(f"{ml_model},{imbalance_method}:")
        display(df_results)
        print("\n")


svm_linear,cost_sensitive:


,margin_threshold,total_samples_in_X_inside_margin,total_samples_in_X_outside_margin,total_faults_in_X_inside_margin,total_faults_in_X_outside_margin,precision_class_1_X_outside_margin,recall_class_1_X_outside_margin,f1_class_1_X_outside_margin
0,"(-0.1, 0.1)",5,1004,2,44,0.596774,0.840909,0.698113
0,"(-0.2, 0.2)",11,998,3,43,0.610169,0.837209,0.705882
0,"(-0.3, 0.3)",16,993,7,39,0.589286,0.846154,0.694737
0,"(-0.4, 0.4)",20,989,9,37,0.581818,0.864865,0.695652
0,"(-0.5, 0.5)",21,988,9,37,0.581818,0.864865,0.695652




svm_linear,smote_plus_cost_sensitive:


,margin_threshold,total_samples_in_X_inside_margin,total_samples_in_X_outside_margin,total_faults_in_X_inside_margin,total_faults_in_X_outside_margin,precision_class_1_X_outside_margin,recall_class_1_X_outside_margin,f1_class_1_X_outside_margin
0,"(-0.1, 0.1)",6,1003,2,44,0.587302,0.840909,0.691589
0,"(-0.2, 0.2)",10,999,4,42,0.583333,0.833333,0.686275
0,"(-0.3, 0.3)",14,995,6,40,0.578947,0.825000,0.680412
0,"(-0.4, 0.4)",18,991,8,38,0.581818,0.842105,0.688172
0,"(-0.5, 0.5)",22,987,10,36,0.574074,0.861111,0.688889




svm_rbf,cost_sensitive:


,margin_threshold,total_samples_in_X_inside_margin,total_samples_in_X_outside_margin,total_faults_in_X_inside_margin,total_faults_in_X_outside_margin,precision_class_1_X_outside_margin,recall_class_1_X_outside_margin,f1_class_1_X_outside_margin
0,"(-0.1, 0.1)",23,986,4,42,0.52,0.619048,0.565217
0,"(-0.2, 0.2)",74,935,26,20,1.00,0.450000,0.620690
0,"(-0.3, 0.3)",90,919,32,14,1.00,0.428571,0.600000
0,"(-0.4, 0.4)",109,900,36,10,1.00,0.500000,0.666667
0,"(-0.5, 0.5)",125,884,37,9,1.00,0.555556,0.714286


... (2 outputs left out)



svm_ploy,cost_sensitive:


,margin_threshold,total_samples_in_X_inside_margin,total_samples_in_X_outside_margin,total_faults_in_X_inside_margin,total_faults_in_X_outside_margin,precision_class_1_X_outside_margin,recall_class_1_X_outside_margin,f1_class_1_X_outside_margin
0,"(-0.1, 0.1)",1,1008,0,46,0.666667,0.173913,0.275862
0,"(-0.2, 0.2)",2,1007,1,45,0.636364,0.155556,0.250000
0,"(-0.3, 0.3)",2,1007,1,45,0.636364,0.155556,0.250000
0,"(-0.4, 0.4)",2,1007,1,45,0.636364,0.155556,0.250000
0,"(-0.5, 0.5)",2,1007,1,45,0.636364,0.155556,0.250000




svm_ploy,smote_plus_cost_sensitive:


,margin_threshold,total_samples_in_X_inside_margin,total_samples_in_X_outside_margin,total_faults_in_X_inside_margin,total_faults_in_X_outside_margin,precision_class_1_X_outside_margin,recall_class_1_X_outside_margin,f1_class_1_X_outside_margin
0,"(-0.1, 0.1)",5,1004,2,44,0.769231,0.909091,0.833333
0,"(-0.2, 0.2)",8,1001,4,42,0.775510,0.904762,0.835165
0,"(-0.3, 0.3)",12,997,5,41,0.787234,0.902439,0.840909
0,"(-0.4, 0.4)",14,995,5,41,0.804348,0.902439,0.850575
0,"(-0.5, 0.5)",24,985,5,41,0.860465,0.902439,0.880952


In [15]:
print(ml_model, imbalance_method)
df_results

svm_ploy smote_plus_cost_sensitive


,margin_threshold,total_samples_in_X_inside_margin,total_samples_in_X_outside_margin,total_faults_in_X_inside_margin,total_faults_in_X_outside_margin,precision_class_1_X_outside_margin,recall_class_1_X_outside_margin,f1_class_1_X_outside_margin
0,"(-0.1, 0.1)",5,1004,2,44,0.769231,0.909091,0.833333
0,"(-0.2, 0.2)",8,1001,4,42,0.775510,0.904762,0.835165
0,"(-0.3, 0.3)",12,997,5,41,0.787234,0.902439,0.840909
0,"(-0.4, 0.4)",14,995,5,41,0.804348,0.902439,0.850575
0,"(-0.5, 0.5)",24,985,5,41,0.860465,0.902439,0.880952


In [16]:
# step2: Goal (summarize results):
# For all models:
# - for a given threshold  
# - compute labeling cost (total_samples_in_X_inside margin) 
# - compute F1-score (f1_class_1_X_outside_margin)

In [19]:
# create dataframe structure to summarize results
rows = pd.MultiIndex.from_product(
        [ml_models_svm],
        names=["ML model"]
    )    
# column
cols = pd.MultiIndex.from_product(
        [imbalance_learn_methods_svm, ['f1_score_X_out','cost']]
    )
    
summary_table = pd.DataFrame(index=rows, columns=cols)

margin_threshold = (-0.5,0.5)

for ml_model in ml_models_svm:
    for imbalance_method in imbalance_learn_methods_svm:
        out_dir = results_dir / 'post_deployment_performance_handling'
        #print(ml_model,imbalance_method)
        fname = f'{out_dir}/results_{ml_model}_{imbalance_method}.h5'
        #print(fname)
        df_results = pd.read_hdf(fname)
        df_results = df_results[df_results['margin_threshold']==margin_threshold]
        #print(df_results)
        f1_score_X_outside_margin = df_results['f1_class_1_X_outside_margin'].values
        cost = df_results['total_samples_in_X_inside_margin'].values

        summary_table.loc[(ml_model), (imbalance_method, 'f1_score_X_out')] =f1_score_X_outside_margin 
        summary_table.loc[(ml_model), (imbalance_method, 'cost')] =cost

#save summary
summary_table = summary_table.round(2)
summary_table.to_csv(f"{out_dir}/summary.csv",  float_format="%.2f")

#display
pd.options.display.float_format = '{:.2f}'.format
display(summary_table)       

cost_sensitive      smote_plus_cost_sensitive     
           f1_score_X_out cost            f1_score_X_out cost
ML model                                                     
svm_linear           0.70   21                      0.69   22
svm_rbf              0.71  125                      0.98   34
svm_ploy             0.25    2                      0.88   24